**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Multiple Tools

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Normally

In [3]:
from langchain_openai import ChatOpenAI

In [4]:
model = ChatOpenAI(model_name='gpt-6-luna')

In [5]:
from langchain_core.prompts import ChatPromptTemplate

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("human", "{input}"),
    ]
)

### Phone Number

In [7]:
msg = prompt.format_prompt(input="전재영의 전화번호를 알려주세요.")

In [8]:
response = model.invoke(msg)

In [9]:
print(response.content)

전재영 씨의 개인 전화번호는 제공해 드릴 수 없습니다. 업무상 연락이 필요하시다면 소속 기관이나 공식 홈페이지에 공개된 연락처를 이용해 주세요.


### Weather

In [10]:
msg = prompt.format_prompt(input="오늘 포항의 날씨를 알려주세요.")

In [11]:
response = model.invoke(msg)

In [12]:
print(response.content)

현재 실시간 날씨를 조회할 수 없어 오늘 포항의 기온이나 강수 여부를 정확히 알려드리기 어렵습니다. **기상청 날씨누리(weather.go.kr)**에서 ‘포항’을 검색하면 최신 예보를 확인하실 수 있어요.


## With Tool Calling

### Make Tools

In [13]:
from langchain_core.tools import tool

In [14]:
@tool
def find_phone_number(name: str) -> str:
    """
    주어진 사람의 전화번호를 찾아서 반환합니다.
    전화번호를 찾지 못한 경우 None을 반환합니다.
    """

    phone_book = {
        "전재영": "917-555-0316",
        "홍길동": "718-333-2232",
        "김미선": "613-123-4434",
    }

    if name in phone_book:
        return phone_book[name]
    else:
        return None

In [15]:
@tool
def get_weather(location: str) -> str:
    """
    주어진 지역에 대한 날씨를 반환합니다.
    날씨 정보가 없는 경우 None을 반환합니다.
    """

    weather_data = {
        "서울": "맑음",
        "포항": "바람 많이 분다",
        "제주도": "비옴",
    }

    if location in weather_data:
        return weather_data[location]
    else:
        return None

### A Collection of Tools

In [16]:
tools = [
    find_phone_number,
    get_weather
]

### Set Up Tool Calling Agent

In [17]:
from langchain.agents import create_agent

In [18]:
agent = create_agent(model, tools, system_prompt="You are a helpful assistant")

### Invoke

#### Example 1

In [19]:
response = agent.invoke(
    {
        "messages": [("user", "전재영의 전화번호를 알려주세요.")]
    }
)

In [20]:
response

{'messages': [HumanMessage(content='전재영의 전화번호를 알려주세요.', additional_kwargs={}, response_metadata={}, id='f61f1d1e-8aec-4efe-868b-521d6b9efe8e'),
  AIMessage(content=[{'arguments': '{"name":"전재영"}', 'call_id': 'call_IDcTn1U9WivPRUSKHSoguIa1', 'name': 'find_phone_number', 'type': 'function_call', 'id': 'fc_00ce88b503d2e85c006af72bd24d5c87d29361ef882c0a2673', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00ce88b503d2e85c006af72bd0e2e887d2a0d7783794bf5b4c', 'created_at': 1794583504.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00ce88b503d2e85c006af72bd0e2e887d2a0d7783794bf5b4c', tool_calls=[{'name': 'find_phone_number', 'args': {'name': '전재영'}, 'id': 'call_IDcTn1U9WivPRUSKHSoguIa1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 21, 'total_tokens': 138, 'input_token_details':

#### Example 2

In [21]:
response = agent.invoke(
    {
        "messages": [("user", "오늘 포항 날씨를 알려주세요.")]
    }
)

In [22]:
print(response["messages"][-1].content)

[{'id': 'rs_09e86e2be4ec9156006af72bd8aeb487d2a9672e031f17dd64', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq9yvZO9hmSfsbVBZC8UUw8hrz7ZjT3mRtBDj0p1fusvHaFpxuztQG3XtQFSyFprBEujJqtuDrtV9a-_vk51prhVn-OoRtCvYcUCuUTjsmhBfkD30yTMIafbQ82MQdMGIkUkhZWhitqQVTF26SYrTmCndopReq7vM9Lk4igiuPEPrQJRYMLRaBSUzjPCL2qOFC186YdGj2q2BlOQSQY5bNZIjYzl1MIDpeQTXhFQXbZuUHh4aPdQusGt_6mtLnkKeHEC3RmpvWL4xSwX-pKqU5rDFt3fXlDeRNsIvIS9wiPtBUwXONpL1qdIaHLn24xfdEXVQBwSUncXfnG25xR2baYnfNINf_FNYRqZq-3NuyxvGOTbO8oCx6DAi9WXgCnO6sqDyAMHwCiWSjHp_eCg13zucPCe3HU83d65xvaZdrvziHKt5RKB-x-Qeim_Apjc1DVoCG3_lr6aqdksWo5Yy-vOdrXRu-gn6gweQKkjOdUXhVDMOs_FjoKD3cSkw-V52oYmg3C1ni_kwwOGLCfy6Ik9abQxsu7DGia1RV_YRgvuPVIC1062EDnGLVcIsWdl1XtgzLryzK-DofPfac2UlX5ugHKf1UepKReGXl_-qME_z-uMsASSXxv4HJjOx_uKeHcgz85wfjgSIRZMedh383dRnXAV2R5sWt2m7zuXIokBSvwYojwCRsexktycarDJikBgJWaPq89Hy0CvszPXxoYoJzCCZs6r30zfWs3EnrCk_NjcUYW23zJEqmlUJKz8YnU3LJ3WMJbBw6_DrkFnL5waYNglHptoqm9WOj_q853OSmzOdvD1q2fkNy3HKoNXfHC2Do6XS-WHkxZMB7W0jvzt

#### Example 3

In [23]:
response = agent.invoke(
    {
        "messages": [("user", "전재영의 전화번호와 서울의 날씨를 알려주세요.")]
    }
)

In [24]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '전재영의 전화번호는 917-555-0316이고, 서울은 맑습니다.', 'annotations': [], 'id': 'msg_0740f2c15f89a4ce006af72bdd8c3087d2a3ba53e8fa937bdf', 'phase': 'final_answer'}]


#### Example 4

In [25]:
response = agent.invoke(
    {
        "messages": [("user", "김요한의 전화번호와 뉴욕의 날씨를 알려주세요.")]
    }
)

In [26]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '김요한의 전화번호와 뉴욕의 날씨 정보를 찾을 수 없습니다.', 'annotations': [], 'id': 'msg_0633f2e7602694e5006af72be1bbec87d299a818772f04771b', 'phase': 'final_answer'}]
